In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/Datasets/home_credit_final_table.csv', sep=';')
print(df.head())
print(df.shape)

   TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY  \
0       0         Cash loans           M            Y               Y   
1       0         Cash loans           M            N               Y   
2       0         Cash loans           M            Y               N   
3       1         Cash loans           F            N               N   
4       0         Cash loans           F            N               Y   

   CNT_CHILDREN  AMT_INCOME_TOTAL  AMT_CREDIT  AMT_ANNUITY  AMT_GOODS_PRICE  \
0             3          135000.0    373140.0      25065.0         337500.0   
1             0          225000.0    675000.0      17937.0         675000.0   
2             1          270000.0    521280.0      31500.0         450000.0   
3             0          135000.0    288873.0      16258.5         238500.0   
4             0          135000.0    364896.0      19926.0         315000.0   

   ... share_zero_payment_pa avg_cnt_payment_pa share_yield_unknown_pa  \
0  ...      

In [3]:
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
print(missing_pct)

# Identifying columns with missing values and their share of all oberservations
# Output implies that only categorical columns have missing values, further investigation is required

FONDKAPREMONT_MODE     68.386172
WALLSMATERIAL_MODE     50.840783
HOUSETYPE_MODE         50.176091
EMERGENCYSTATE_MODE    47.398304
OCCUPATION_TYPE        31.345545
NAME_TYPE_SUITE         0.420148
dtype: float64


In [4]:
df[missing_pct.index].describe(include='all')

# Gathering information on columns with missing values
# Output shows that the columns are all categorical

,FONDKAPREMONT_MODE,WALLSMATERIAL_MODE,HOUSETYPE_MODE,EMERGENCYSTATE_MODE,OCCUPATION_TYPE,NAME_TYPE_SUITE
count,97216,151170,153214,161756,211120,306219
unique,4,7,3,2,18,7
top,reg oper account,Panel,block of flats,No,Laborers,Unaccompanied
freq,73830,66040,150503,159428,55186,248526


In [5]:
df.fillna('Unknown', inplace=True)

# All columns are categorical, which allows to replace all missing values with value "Unknown"

In [6]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['TARGET'])
y = df['TARGET']

X = pd.get_dummies(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=87, stratify=y)
X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train, y_train, train_size=0.3, random_state=87, stratify=y_train
)

print(X_train.shape, X_test.shape)
print(X_train_sample.shape, X_test.shape)

# Splitting the dataframe into 80% training set and 20% test set
# Given the large dataset size (~ 300k observations), a simple train-test split is sufficient
# stratify=y insured that the class distribution of the binary target variable is identical in train and test split
# An additional 30% sample of the training set is created to reduce training time

(246008, 392) (61503, 392)
(73802, 392) (61503, 392)


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler1 = StandardScaler()
X_train_sample_scaled = scaler1.fit_transform(X_train_sample)
X_test_scaled1 = scaler1.transform(X_test)

lr_model1 = LogisticRegression(random_state=87, max_iter = 1000, class_weight = 'balanced')
lr_model1.fit(X_train_sample_scaled, y_train_sample)

# Baseline model: Logistic Regression trained on the 30% sample training set
# StandardScaler is introduced to standardize it's features
# If there was no StandardScaler, due to active L2 regularization, features with large betas would be penalized unfairly
# class_weight='balanced' adjusts weights inversely proportional to class frequency,
# compensating for the class imbalance (~8% defaults)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=87)

In [8]:
from sklearn.metrics import roc_auc_score, classification_report

y_pred = lr_model1.predict(X_test_scaled1)
y_pred_proba = lr_model1.predict_proba(X_test_scaled1)[:, 1]

print("ROC_AUC:", roc_auc_score(y_test, y_pred_proba))
print(classification_report(y_test, y_pred))

# Evaluating lr_model1 on the test set
# ROC-AUC is chosen as primary metric as it is robust to class imbalance - unlike accuracy,
# which would be misleading given only ~8% defaults in the dataset
# classification_report additionally shows precision/recall per class

ROC_AUC: 0.7551938136270103
              precision    recall  f1-score   support

           0       0.96      0.70      0.81     56538
           1       0.17      0.68      0.27      4965

    accuracy                           0.70     61503
   macro avg       0.56      0.69      0.54     61503
weighted avg       0.90      0.70      0.77     61503



In [9]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np

sample_weights = compute_sample_weight(class_weight = 'balanced', y = y_train_sample)

# Preparing GBM model
# compute_sample_weight replicates the effect of class_weight='balanced' from Logistic Regression

In [10]:
# param_grid_gbm1 = {
#     'n_estimators': [100, 500, 1000],
#     'learning_rate': [0.01, 0.05, 0.1],
#     'max_depth': [3, 5, 7],
# }

# gbm_modelrs1 = GradientBoostingClassifier(
#     random_state=87,
#     n_iter_no_change=15,
#     validation_fraction=0.1,
#     tol=1e-4
# )

# random_search = RandomizedSearchCV(
#     gbm_modelrs1,
#     param_distributions=param_grid_gbm1,
#     n_iter=15,
#     cv=3,
#     scoring='roc_auc',
#     n_jobs=-1,
#     random_state=87
# )

# random_search.fit(X_train_sample, y_train_sample, sample_weight = sample_weights)

# print("Best hyperparameters:", random_search.best_params_)
# print("Best ROC-AUC score:", random_search.best_score_)

# Output:
# Best hyperparameters: {'n_estimators': 1000, 'max_depth': 3, 'learning_rate': 0.05}
# Best ROC-AUC score: 0.767870665282993

# This code is commented out because it took 7hrs to run and is thus not practical for repetitive execution
# Hyperparameter search was conducted on the 30% sample training set to reduce computation time

In [11]:
gbm_model1 = GradientBoostingClassifier(
    n_estimators= 1000,
    max_depth = 3,
    learning_rate = 0.05,
    random_state=87
)

gbm_model1.fit(X_train_sample, y_train_sample, sample_weight = sample_weights)

y_pred_proba = gbm_model1.predict_proba(X_test)[:, 1]
y_pred = gbm_model1.predict(X_test)

print("ROC_AUC:", roc_auc_score(y_test, y_pred_proba))
print(classification_report(y_test, y_pred))

# Final GBM model trained with optimal hyperparameters found in the search above
# Trained on the 30% sample training set to match the conditions of the hyperparameter search
# Evaluated on the full test set for comparison with other models
# Output shows an increase of ROC_AUC compared to lr_model1

ROC_AUC: 0.7701324069149083
              precision    recall  f1-score   support

           0       0.96      0.76      0.85     56538
           1       0.19      0.64      0.29      4965

    accuracy                           0.75     61503
   macro avg       0.58      0.70      0.57     61503
weighted avg       0.90      0.75      0.80     61503



In [12]:
import xgboost as xgb

In [13]:
# param_grid_xgb1 = {
#     'n_estimators': [100, 500, 1000],
#     'learning_rate': [0.01, 0.05, 0.1],
#     'max_depth': [3, 5, 7],
#     'gamma': [0, 0.1, 0.5],
#     'reg_lambda': [0, 1, 5, 10]
# }

# xgb_modelrs1 = xgb.XGBClassifier(
#     random_state=87,
#     eval_metric='auc',
#     n_jobs=-1
# )

# random_search_xgb1 = RandomizedSearchCV(
#     xgb_modelrs1,
#     param_distributions=param_grid_xgb1,
#     n_iter=15,
#     cv=3,
#     scoring='roc_auc',
#     n_jobs=-1,
#     random_state=87
# )

# random_search_xgb1.fit(X_train_sample, y_train_sample, sample_weight=sample_weights)

# print("Best hyperparameters:", random_search_xgb1.best_params_)
# print("Best ROC_AUC score:", random_search_xgb1.best_score_)

# Output:
# Best hyperparameters: {'reg_lambda': 5, 'n_estimators': 500, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0.1}
# Best ROC_AUC score: 0.7749116899569706

# Hyperparameter search for XGBoost on the 30% sample training set to ensure comparability with GBM search
# Commented out due to long runtime
# Note: despite gamma = 0.1 and reg_lambda = 5 introducing additional regularization,
# the search found fewer n_estimators than GBM, which is likely a coincidence of the random search with only 15 iterations

In [14]:
xgb_model1 = xgb.XGBClassifier(
    n_estimators = 500,
    learning_rate = 0.05,
    max_depth = 3,
    gamma = 0.1,
    reg_lambda = 5,
    random_state=87,
    eval_metric='auc',
    n_jobs=-1
)

xgb_model1.fit(X_train_sample, y_train_sample, sample_weight=sample_weights)

y_pred_proba = xgb_model1.predict_proba(X_test)[:, 1]
y_pred = xgb_model1.predict(X_test)

print("ROC_AUC:", roc_auc_score(y_test, y_pred_proba))
print(classification_report(y_test, y_pred))

# XGBoost model trained with optimal hyperparameters from the search above
# Trained on the 30% sample training set to ensure comparability with gbm_model1
# Evaluated on the full test set, output shows slight AUC improvement over gbm_model1

ROC_AUC: 0.771912149060545
              precision    recall  f1-score   support

           0       0.96      0.74      0.84     56538
           1       0.18      0.67      0.29      4965

    accuracy                           0.73     61503
   macro avg       0.57      0.70      0.56     61503
weighted avg       0.90      0.73      0.79     61503



In [15]:
scaler2 = StandardScaler()
X_train_scaled = scaler2.fit_transform(X_train)
X_test_scaled2 = scaler2.transform(X_test)

lr_model2 = LogisticRegression(random_state=87, max_iter = 1000, class_weight = 'balanced')
lr_model2.fit(X_train_scaled, y_train)

# Second Logistic Regression trained on the full training set
# Feasible due to fast training time of Logistic Regression compared to GBM/XGBoost
# Allows direct comparison of sample vs. full training data impact on model performance

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=87)

In [16]:
y_pred = lr_model2.predict(X_test_scaled2)
y_pred_proba = lr_model2.predict_proba(X_test_scaled2)[:, 1]

print("ROC_AUC:", roc_auc_score(y_test, y_pred_proba))
print(classification_report(y_test, y_pred))

# Evaluation of lr_model2 on the test set
# Despite training on the full dataset, ROC_AUC is lower than gbm_model1 and xgb_model1,
# which were only trained on 30% of the training data
# this suggests GBM and XGBoost are fundamentally superior models for this classification problem

ROC_AUC: 0.76268821792877
              precision    recall  f1-score   support

           0       0.96      0.70      0.81     56538
           1       0.17      0.70      0.27      4965

    accuracy                           0.70     61503
   macro avg       0.57      0.70      0.54     61503
weighted avg       0.90      0.70      0.76     61503



In [17]:
weights = compute_sample_weight(class_weight = 'balanced', y = y_train)

# Computing sample weights for the full training set
# Used for xgb_model2, which is trained on the full training data (similiar to sample_weights for the 30% sample)

In [18]:
# param_grid_xgb2 = {
#     'n_estimators': [500, 1000, 3000],
#     'learning_rate': [0.005, 0.01, 0.05, 0.1],
#     'max_depth': [3, 5, 7],
#     'gamma': [0, 0.1, 0.5],
#     'reg_lambda': [0, 1, 5, 10]
# }
#
# xgb_modelrs2 = xgb.XGBClassifier(
#     random_state=87,
#     eval_metric='auc',
#     n_jobs=-1
# )
#
# random_search_xgb2 = RandomizedSearchCV(
#     xgb_modelrs2,
#     param_distributions=param_grid_xgb2,
#     n_iter=30,
#     cv=3,
#     scoring='roc_auc',
#     n_jobs=-1,
#     random_state=87
# )
#
# random_search_xgb2.fit(X_train, y_train, sample_weight=weights)
#
# print("Best hyperparameters:", random_search_xgb2.best_params_)
# print("Best ROC_AUC score:", random_search_xgb2.best_score_)
#
# Output:
# Best hyperparameters: {'reg_lambda': 5, 'n_estimators': 3000, 'max_depth': 3, 'learning_rate': 0.05, 'gamma': 0}
# Best ROC_AUC score: 0.7824256681315657

# Hyperparameter search for XGBoost on the full training set with an expanded parameter grid
# n_iter=30 and larger n_estimators range (up to 3000) compared to xgb_model1 search
# Commented out due to runtime of 8+ hours
# n_estimators=3000 selected as optimal. This is hitting the upper grid boundary,
# suggesting further improvement may be possible with larger n_estimators

In [19]:
xgb_model2 = xgb.XGBClassifier(
    n_estimators = 3000,
    learning_rate = 0.05,
    max_depth = 3,
    gamma = 0,
    reg_lambda = 5,
    random_state=87,
    eval_metric='auc',
    n_jobs=-1
)

xgb_model2.fit(X_train, y_train, sample_weight=weights)

y_pred_proba = xgb_model2.predict_proba(X_test)[:, 1]
y_pred = xgb_model2.predict(X_test)

print("ROC_AUC:", roc_auc_score(y_test, y_pred_proba))
print(classification_report(y_test, y_pred))

# Final and best model in this pipeline, trained on the full training set with optimal hyperparameters
# Achieves the highest ROC_AUC of all models, with a slight decrease in recall for class 1 vs. Logistic Regression
# lowering the classification threshold below 0.5 would increase recall for class 1 (fewer missed defaults)
# at the cost of more false positives, a business decision depending on the relative cost of each error type

ROC_AUC: 0.7821957476790112
              precision    recall  f1-score   support

           0       0.96      0.76      0.85     56538
           1       0.19      0.65      0.30      4965

    accuracy                           0.75     61503
   macro avg       0.58      0.71      0.57     61503
weighted avg       0.90      0.75      0.80     61503

